In [26]:
import torch
import torch.nn as nn

In [27]:
block_size = 8 # n-1 gram
embedding_dim = 16
layer_size = 512
batch_size = 512
steps = 20000

In [28]:
# input & output initialization
with open("inputs-shakespeare.txt", "r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(chars)}
itoc = dict(enumerate(chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, block_size, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[block_size:]], dtype=torch.long)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, block_size, 1)
y_val = torch.tensor([ctoi[ch] for ch in val[block_size:]], dtype=torch.long)

In [29]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, block_size, vocab_size, embedding_dim, layer_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Embedding(vocab_size, embedding_dim),
            nn.Flatten(),
            nn.Linear(in_features=embedding_dim * block_size, out_features=layer_size),
            nn.ReLU(),
            nn.Linear(in_features=layer_size, out_features=vocab_size)
        )

    def forward(self, x):
        return self.layers(x)

In [30]:
# model initialization
model = MultiLayerPerceptron(block_size, vocab_size, embedding_dim, layer_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [31]:
# training
for step in range(steps):
    optimizer.zero_grad()

    ix = torch.randint(0, len(x_train), (batch_size,))
    logits_train = model(x_train[ix])
    loss_train = nn.functional.cross_entropy(logits_train, y_train[ix])

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            logits_val = model(x_val)
            loss_val = nn.functional.cross_entropy(logits_val, y_val)
            print(f"Step {step}: train loss = {loss_train.item()}, val loss = {loss_val.item()}")

    loss_train.backward()
    optimizer.step()

Step 0: train loss = 4.2344489097595215, val loss = 4.221679210662842
Step 1000: train loss = 1.85531485080719, val loss = 1.9271647930145264
Step 2000: train loss = 1.7010542154312134, val loss = 1.8313745260238647
Step 3000: train loss = 1.5299540758132935, val loss = 1.7902965545654297
Step 4000: train loss = 1.4494946002960205, val loss = 1.788103461265564
Step 5000: train loss = 1.5741900205612183, val loss = 1.7718104124069214
Step 6000: train loss = 1.5599102973937988, val loss = 1.76462721824646
Step 7000: train loss = 1.5350396633148193, val loss = 1.7521743774414062
Step 8000: train loss = 1.4710732698440552, val loss = 1.7504584789276123
Step 9000: train loss = 1.4788278341293335, val loss = 1.7470453977584839
Step 10000: train loss = 1.477095365524292, val loss = 1.7454570531845093
Step 11000: train loss = 1.3986539840698242, val loss = 1.737268090248108
Step 12000: train loss = 1.371503472328186, val loss = 1.7258079051971436
Step 13000: train loss = 1.4940273761749268, va

In [32]:
# prediction
with torch.inference_mode():
    output = ['\n'] * block_size
    max_output_length = 1000
    while len(output) < max_output_length:
        i = torch.tensor([[ctoi[ch] for ch in output[-block_size:]]], dtype=torch.long)
        
        logits = model(i)
        P = torch.softmax(logits, dim=1)

        j = torch.multinomial(P, num_samples=1, replacement=True).item()

        output.append(itoc[j])

In [33]:
print(''.join(output[block_size:]))

POLIXENES:
Doze to seek sweet sin friar of our reconcration friens:
They prison!
Can early enemy at, after of e'bres
More some wear
But I cannot us sograble.

WARWICK:
Ay, By no shilled?

AUTOLYCUS:
Musty wit thereto werel quick leave;
If I disterance.
What despect to her in Aut, by anilys Keep we do the words of not saw so the stell thyself
Atten, sweet should be I live were with some,
In his flil's this world by for thence, it would thy desirih
Of as criviles withdenon of a sorrab worthy hand:
They, thy detent from at my and to be possevel
To be not, by the comes a cangern to unit, and all you to be so, my lord way, it sworn noise: his prince.'

Froy prich? Richard wish the king,
And this thrating of ourding compwarney,
Turnstio. Gain of:
That valiant, and so in this rowert on a brow to one close musicien? On heretale her tage
She's very galivest shortent onters: be you ion the corphords forlow Romen, so fany to shapitarny a member,
When I cannot consent.
Kist you went
Whereo
